<a href="https://colab.research.google.com/github/andilMc/gemmafro-e2b/blob/main/notebooks/05_generate_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Génération de la soumission finale (Phase 6)

Correspond à la Phase 6 de `docs/WORKFLOW.md`. Charge l'adaptateur LoRA final, génère les réponses pour toutes les questions de `Test.csv`, et construit le fichier de soumission au format attendu par Zindi.

In [1]:
# Installe les dépendances.
!pip install -q -U transformers accelerate peft bitsandbytes datasets pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 132.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 w

In [2]:
# Monte Drive, localise l'adaptateur LoRA et crée le dossier des soumissions.
import os
from google.colab import drive

drive.mount('/content/drive')  # demande l'autorisation d'accès au Drive

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'  # racine du projet sur Drive
ADAPTER_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'  # adaptateur LoRA final de la Phase 3
SUBMISSIONS_DIR = f'{PROJECT_DIR}/submissions'  # dossier des fichiers à soumettre
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)  # crée le dossier s'il n'existe pas
# arrête tôt si la Phase 3 n'a pas été exécutée
assert os.path.isdir(ADAPTER_DIR), "Adaptateur introuvable — exécuter 03_finetune.ipynb jusqu'au bout d'abord."

Mounted at /content/drive


In [3]:
# Se connecte à Hugging Face avec le token des Colab Secrets.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))  # authentifie la session avec le token HF_TOKEN

In [4]:
# Charge Test.csv (questions sans réponse) et SampleSubmission.csv (format attendu par Zindi).
import pandas as pd

REPO_DIR = '/content/gemmafro-e2b'
if not os.path.isdir(REPO_DIR):  # clone seulement si le dépôt est absent
    # télécharge le dépôt du projet
    !git clone https://github.com/andilMc/gemmafro-e2b.git {REPO_DIR}

test = pd.read_csv(f'{REPO_DIR}/data/Test.csv')  # questions de test
sample_submission = pd.read_csv(f'{REPO_DIR}/data/SampleSubmission.csv')  # modèle de soumission, pour vérifier les ID
print(test.shape)
test.head(3)

Cloning into '/content/gemmafro-e2b'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 150 (delta 89), reused 80 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 6.23 MiB | 22.53 MiB/s, done.
Resolving deltas: 100% (89/89), done.
(2618, 3)


,ID,input,subset
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwu...",Aka_Gha
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ w...,Aka_Gha
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusua...,Aka_Gha


In [5]:
# Charge le modèle de base en 4-bit et l'adaptateur LoRA, comme dans 04_evaluate.ipynb : mêmes réglages
# 4-bit/bf16 et même déballage Gemma4ClippableLinear avant l'adaptateur.
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "google/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512  # même longueur maximale qu'à l'entraînement

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer Gemma
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # pas de token de padding défini : on réutilise le token de fin
tokenizer.padding_side = 'left'  # padding à gauche pour la génération par batch

gc.collect()  # libère la mémoire (Python, puis cache GPU)
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(  # même quantification 4-bit qu'à l'entraînement
    load_in_4bit=True,  # quantifie les poids en 4 bits pour tenir en VRAM
    bnb_4bit_quant_type="nf4",  # format NormalFloat4, adapté aux poids de réseaux de neurones
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,  # quantifie aussi les constantes de quantification (gain mémoire)
)

base_model = AutoModelForCausalLM.from_pretrained(  # charge le modèle de base quantifié
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,  # poids non quantifiés en bf16
    device_map={"": 0},  # tout sur le GPU 0, sans offload CPU
)
base_model.config.pad_token_id = tokenizer.pad_token_id  # aligne le modèle sur le token de padding du tokenizer
base_model.config.use_cache = True  # cache KV : accélère la génération

def unwrap_clippable_linears(model):
    count = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if child.__class__.__name__ == "Gemma4ClippableLinear":
                setattr(module, child_name, child.linear)
                count += 1
    print(f'{count} couches Gemma4ClippableLinear déballées vers Linear4bit')
    return model

base_model = unwrap_clippable_linears(base_model)  # déballage avant de charger l'adaptateur

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)  # rattache les poids LoRA au modèle de base
model.eval()  # mode inférence (désactive le dropout)

gc.collect()  # libère la mémoire (Python, puis cache GPU)
torch.cuda.empty_cache()
print(f"Mémoire GPU allouée : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

232 couches Gemma4ClippableLinear déballées vers Linear4bit
Mémoire GPU allouée : 6.86 Go


In [6]:
# Construit les prompts de test avec le même format qu'en Phases 2 et 3 : nom de langue, question et chat
# template.
SUBSET_TO_LANGUAGE = {  # préfixe du code subset → nom de la langue
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    if not subset_code or not isinstance(subset_code, str):  # valeur manquante ou invalide : anglais par défaut
        return 'English'
    # garde le code brut si la langue est inconnue
    return SUBSET_TO_LANGUAGE.get(subset_code.split('_')[0], subset_code)

def build_prompt(question: str, language: str) -> str:  # même instruction qu'en Phase 2
    return (
        f"Réponds à la question de santé suivante en {language}, "
        f"de façon claire et médicalement fiable.\n\nQuestion : {question}"
    )

test['prompt_text'] = test.apply(  # prompt formaté de chaque question de test
    lambda row: tokenizer.apply_chat_template(
        [{"role": "user", "content": build_prompt(row['input'], subset_to_language_name(row['subset']))}],
        tokenize=False, add_generation_prompt=True,  # texte brut avec l'amorce de la réponse
    ),
    axis=1,  # applique la fonction ligne par ligne
)

In [7]:
# Génère la réponse à chaque question de Test.csv, par batch (décodage déterministe, 400 tokens max).
import re  # sert à nettoyer les prédictions à l'étape suivante

@torch.no_grad()  # pas de calcul de gradient : économise la mémoire
def generate_answers(prompts, batch_size=8, max_new_tokens=400):  # génère les réponses par batch
    answers = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(  # tokenise le batch de prompts
            batch, return_tensors='pt', padding=True, truncation=True,  # tenseurs PyTorch, avec padding et troncature
            max_length=MAX_SEQ_LENGTH, add_special_tokens=False,  # tronque les prompts trop longs
        ).to(model.device)  # envoie le batch sur le GPU
        out = model.generate(  # génère la suite du prompt
            **inputs,
            max_new_tokens=max_new_tokens,  # longueur maximale de la réponse
            do_sample=False,  # décodage déterministe (greedy)
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = out[:, inputs['input_ids'].shape[1]:]  # retire le prompt : ne garde que les tokens générés
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)  # tokens → texte, sans tokens spéciaux
        answers.extend(d.strip() for d in decoded)
        if (i // batch_size) % 10 == 0:  # affiche la progression tous les 10 batchs
            print(f'{min(i + batch_size, len(prompts))}/{len(prompts)}')
    return answers

print(f'Génération pour {len(test)} questions de test...')
# une réponse par question de test, dans le même ordre
test_predictions = generate_answers(test['prompt_text'].tolist())
print('\n✅ Génération terminée')

Génération pour 2618 questions de test...
8/2618
88/2618
168/2618
248/2618
328/2618
408/2618
488/2618
568/2618
648/2618
728/2618
808/2618
888/2618
968/2618
1048/2618
1128/2618
1208/2618
1288/2618
1368/2618
1448/2618
1528/2618
1608/2618
1688/2618
1768/2618
1848/2618
1928/2618
2008/2618
2088/2618
2168/2618
2248/2618
2328/2618
2408/2618
2488/2618
2568/2618

✅ Génération terminée


In [8]:
# Construit le fichier de soumission Zindi (ID + trois colonnes de prédiction) et le valide avant de le
# sauvegarder.
# prépare, vérifie et sauvegarde le fichier de soumission
def make_submission(ids, predictions, output_path, reference_ids):
    # Belt-and-suspenders : retire d'éventuels tokens spéciaux résiduels.
    clean_preds = [re.sub(r'<[^>]+>', '', str(p)).strip() for p in predictions]

    sub = pd.DataFrame({
        'ID': ids,
        'TargetRLF1': clean_preds,  # même réponse pour les trois cibles évaluées (ROUGE-L, ROUGE-1, LLM)
        'TargetR1F1': clean_preds,
        'TargetLLM': clean_preds,
    })[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]  # ordre des colonnes attendu par Zindi

    # Vérifications avant sauvegarde.
    assert set(sub['ID']) == set(reference_ids), "Les ID ne correspondent pas exactement à Test.csv"
    assert not sub['ID'].duplicated().any(), "IDs dupliqués dans la soumission"
    assert not sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].isna().any().any(), "Valeurs manquantes"
    assert (sub['TargetRLF1'].str.len() > 0).all(), "Réponses vides détectées"

    sub.to_csv(output_path, index=False, encoding='utf-8')  # UTF-8 : préserve les écritures non latines (amharique)
    print(f'✅ Soumission sauvegardée : {output_path} ({len(sub)} lignes)')
    return sub

submission = make_submission(  # génère le fichier dans le dossier submissions/ sur Drive
    test['ID'], test_predictions,
    f'{SUBMISSIONS_DIR}/submission_gemma4_e2b_finetuned.csv',
    sample_submission['ID'],
)
submission.head(5)

✅ Soumission sauvegardée : /content/drive/MyDrive/gemmafro-e2b/submissions/submission_gemma4_e2b_finetuned.csv (2618 lignes)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,Amanneɛbɔ ne Nsɛm a Wɔka Kyerɛ: Amanneɛbɔ ne n...,Amanneɛbɔ ne Nsɛm a Wɔka Kyerɛ: Amanneɛbɔ ne n...,Amanneɛbɔ ne Nsɛm a Wɔka Kyerɛ: Amanneɛbɔ ne n...
1,ID_TS_Aka_Gha_1C80317F,Hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi...,Hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi...,Hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi...
2,ID_TS_Aka_Gha_06671AD1,Mmabun betumi ahyɛ wɔn ho so denam: Nsɛm a wɔb...,Mmabun betumi ahyɛ wɔn ho so denam: Nsɛm a wɔb...,Mmabun betumi ahyɛ wɔn ho so denam: Nsɛm a wɔb...
3,ID_TS_Aka_Gha_BDD640FB,"Amammerɛ mu mmra, asetena mu suban, ne tumi mu...","Amammerɛ mu mmra, asetena mu suban, ne tumi mu...","Amammerɛ mu mmra, asetena mu suban, ne tumi mu..."
4,ID_TS_Aka_Gha_46685257,Mmara nsesaeɛ ho hia ma mmabun nyin wɔ biribia...,Mmara nsesaeɛ ho hia ma mmabun nyin wɔ biribia...,Mmara nsesaeɛ ho hia ma mmabun nyin wɔ biribia...


In [9]:
# Aperçu : une question et sa réponse générée pour chaque langue.
preview = test[['ID', 'subset', 'input']].copy()  # question et langue de chaque ligne de test
preview['answer'] = submission['TargetRLF1']  # réponse générée correspondante

for lang in sorted(preview['subset'].unique()):  # une langue à la fois
    row = preview[preview['subset'] == lang].iloc[0]  # première ligne de cette langue
    print(f"[{lang}] {row['input'][:90]}")
    print(f"  → {row['answer'][:150]}")
    print()

[Aka_Gha] Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwu
  → Amanneɛbɔ ne Nsɛm a Wɔka Kyerɛ: Amanneɛbɔ ne nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano

[Amh_Eth] ክላሚዲያ ሳይታከም ቢቆይ በወንዶች ላይ የሚያስከትለው የረጅም ጊዜ ጉዳት ምንድን ነው?
  → የመካን መፍሰስ ሲያደንቅ፣ የሽንት ቱቦዎች እብጠት (urethritis) እና በከብት መራቢያ አካላት ላይ ጉዳት ሊያደርስ ይችላል።

[Eng_Eth] How can I use the success stories of educated women to inspire girls?
  → This is a question about, Education/Advocacy. Share their stories in school presentations, community meetings, or social media. Show girls that with e

[Eng_Gha] What strategies are effective in promoting awareness, education, and empowerment regarding
  → Effective strategies include: Cultural sensitivity: Tailoring educational materials and approaches to respect diverse cultural beliefs and practices. 

[Eng_Ken] Is there medication available to prevent the contraction of STDs?
  → Yes, there are med

---
**Soumission prête** : `submissions/submission_gemma4_e2b_finetuned.csv` sur Drive. Téléchargez ce fichier et déposez-le sur la page du challenge Zindi pour obtenir le score officiel (`TargetRLF1`, `TargetR1F1`, `TargetLLM`).